# Lab 5b: Adversarial Training With a Malware Classifier

## Lab Objective:

The student will learn how to harden a convolutional neural network against adversarial attacks 
using adversarial training with samples generated by the Fast Gradient Sign Method attack in 
the Cleverhans adversarial library.

## Acknowledgement

This module was developed under the National Science Foundation awards # 2416990, 2416992 and 2607393 at Tennessee Tech University, North Carolina A&T State University and University of Albany.  

## Setup

In [ ]:
# NOTHING TO DO HERE - JUST RUN CELL
# Install needed modules
!pip install kagglehub keras shap tensorflow tensorflow_datasets ipywidgets matplotlib scikit-learn

## Files Needed

This notebook does not embed the data. For local use, keep these paths next to the notebook:

- `samples200/`
- `MalwareClassifier.h5`
- optional backup: `MalwareClassifier.h5.bak`

In Colab, if `samples200/` is missing, the setup cell will prompt you to upload `samples200.zip`. The zip may contain a top-level `samples200/` folder or the eight class folders directly. If `MalwareClassifier.h5` is missing, it will prompt for that file separately.

The samples are inert PNG bytecode texture images, not live malware.

In [ ]:
# Lab 5b runtime setup
import os
import random
import shutil
import zipfile
from pathlib import Path

# Keep TensorFlow logs quieter in notebook output.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from cleverhans.tf2.attacks import fast_gradient_method as fgsm

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path("samples200")
MODEL_PATH = Path("MalwareClassifier.h5")
BACKUP_MODEL_PATH = Path("MalwareClassifier.h5.bak")
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
TRAINING_EPOCHS = 50

LABELS = [
    "Gatak", "Kelihos_ver1", "Kelihos_ver3", "Lollipop",
    "Obfuscator_ACY", "Ramnit", "Tracur", "Vundo"
]

def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise ValueError(f"Unsafe zip member path: {member.filename}")
        archive.extractall(destination)

def normalize_samples_directory(extract_dir):
    extract_dir = Path(extract_dir)
    if DATA_DIR.exists():
        return

    nested = extract_dir / DATA_DIR.name
    if nested.exists():
        shutil.move(str(nested), str(DATA_DIR))
        return

    class_dirs = [extract_dir / label for label in LABELS]
    if all(path.is_dir() for path in class_dirs):
        DATA_DIR.mkdir()
        for path in class_dirs:
            shutil.move(str(path), str(DATA_DIR / path.name))
        return

    raise FileNotFoundError(
        "Could not find samples200/ in the uploaded zip. The zip should contain "
        "a top-level samples200 folder or the eight class folders directly."
    )

def upload_file_if_needed(path, prompt, expected_suffix=None):
    if path.exists():
        print(f"Found {path}.")
        return path
    if not running_in_colab():
        raise FileNotFoundError(
            f"Missing required path: {path}. Run this notebook from inside the 6b folder, "
            "or upload the required file in Colab."
        )

    from google.colab import files
    print(prompt)
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError(f"No file was uploaded for {path}.")

    uploaded_name = next(iter(uploaded))
    uploaded_path = Path(uploaded_name)
    if expected_suffix and uploaded_path.suffix.lower() != expected_suffix:
        raise ValueError(f"Expected a {expected_suffix} file, got {uploaded_name}.")
    return uploaded_path

def ensure_lab_files():
    if not DATA_DIR.exists():
        zip_path = upload_file_if_needed(
            DATA_DIR,
            "Upload samples200.zip. It may contain a top-level samples200 folder, "
            "or the eight malware-family class folders directly.",
            expected_suffix=".zip",
        )
        extract_dir = Path("_samples200_upload")
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        safe_extract_zip(zip_path, extract_dir)
        normalize_samples_directory(extract_dir)
        shutil.rmtree(extract_dir)
        print(f"Extracted {DATA_DIR} from {zip_path}.")
    else:
        print(f"Found {DATA_DIR}.")

    model_upload = upload_file_if_needed(
        MODEL_PATH,
        "Upload MalwareClassifier.h5.",
    )
    if not MODEL_PATH.exists() and model_upload != MODEL_PATH:
        shutil.move(str(model_upload), str(MODEL_PATH))
        print(f"Stored uploaded model as {MODEL_PATH}.")

ensure_lab_files()
print("TensorFlow:", tf.__version__)
print("Keras:", getattr(keras, "__version__", "unknown"))


## Load Helpers

The original script loads a full batch with `ImageDataGenerator`, evaluates the pre-trained model, generates FGSM adversarial samples with CleverHans, trains on those samples, and evaluates again. The notebook keeps that flow but wraps repeated epsilon experiments in functions.

In [ ]:
# Data, model, attack, and reporting helpers.
def count_images_by_class(data_dir=DATA_DIR):
    counts = {}
    for class_dir in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        counts[class_dir.name] = len(list(class_dir.glob("*.png")))
    return counts

def load_dataset(data_dir=DATA_DIR, shuffle=False):
    total_images = sum(count_images_by_class(data_dir).values())
    generator = ImageDataGenerator().flow_from_directory(
        directory=str(data_dir),
        target_size=IMG_SIZE,
        batch_size=total_images,
        shuffle=shuffle,
    )
    images, labels = next(generator)
    images = images.astype("float32") / 255.0
    labels = labels.astype("float32")
    return images, labels, generator.class_indices

def load_compiled_model(model_path=MODEL_PATH):
    # compile=False avoids legacy optimizer-state warnings from the older .h5 export.
    model = keras.models.load_model(model_path, compile=False)
    model.compile(
        optimizer=keras.optimizers.Adam(),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def make_adversarial_examples(model, images, epsilon):
    images_tf = tf.convert_to_tensor(images, dtype=tf.float32)
    return fgsm.fast_gradient_method(model, images_tf, eps=epsilon, norm=np.inf)

def evaluate_accuracy(model, images, labels):
    loss, accuracy = model.evaluate(images, labels, verbose=0)
    return {"loss": float(loss), "accuracy": float(accuracy)}

def format_accuracy(score):
    return f"{score['accuracy'] * 100:.2f}%"

def show_sample(images, labels, index, title):
    class_index = int(np.argmax(labels[index]))
    class_name = LABELS[class_index]
    plt.figure(figsize=(4, 4))
    plt.imshow(np.clip(images[index], 0, 1))
    plt.title(f"{title}\nLabel: {class_name}")
    plt.axis("off")
    plt.show()

def print_report(results):
    print(f"Untrained model, regular data:      {format_accuracy(results['untrained_regular'])}")
    print(f"Untrained model, adversarial data:  {format_accuracy(results['untrained_adversarial'])}")
    print(f"Trained model, adversarial data:    {format_accuracy(results['trained_adversarial'])}")
    print(f"Trained model, regular data:        {format_accuracy(results['trained_regular'])}")


## Load the Dataset

The lab handout describes 200 samples per family. This cell prints the actual counts present in your `samples200/` folder before loading the images.

In [ ]:
# Load all samples from samples200.
class_counts = count_images_by_class()
print("Class counts:")
for class_name, count in class_counts.items():
    print(f"  {class_name:15s} {count}")

x, y, class_indices = load_dataset(shuffle=False)
print("\nClass indices:", class_indices)
print("Dataset shape:", x.shape)


## Step 1: Regular Inputs

Evaluate the pre-trained model on regular, non-adversarial malware sample images. Record the model's accuracy and include a screenshot of the output in your answer document.

In [ ]:
# Load the pre-trained model and evaluate regular inputs.
malware_model = load_compiled_model()
malware_model.summary()

untrained_regular = evaluate_accuracy(malware_model, x, y)
print("Untrained model, regular data:", format_accuracy(untrained_regular))
show_sample(x, y, index=1, title="Non-adversarial sample")


**Student response:**

- Accuracy on regular inputs:
- Screenshot of regular-input accuracy:

## Step 2: Adversarial Inputs Before Training

Generate adversarial samples with FGSM at `epsilon = 0.022`, evaluate the untrained model on those samples, and compare the regular and adversarial sample plots.

In [ ]:
# Generate FGSM adversarial samples for the untrained model.
train_epsilon = 0.022
adv_x = make_adversarial_examples(malware_model, x, train_epsilon)

untrained_adversarial = evaluate_accuracy(malware_model, adv_x, y)
print("Untrained model, adversarial data:", format_accuracy(untrained_adversarial))
show_sample(adv_x.numpy(), y, index=1, title=f"Adversarial sample, epsilon={train_epsilon}")


**Student response:**

- Accuracy on adversarial inputs before adversarial training:
- Screenshot of output accuracies:

## Steps 3 and 4: Train on Adversarial Samples

Train the model on the adversarial samples and evaluate it on both adversarial and regular samples. Some tradeoff is possible because adversarial hardening can change regular-input performance.

In [ ]:
# Train on adversarial samples and evaluate again.
history = malware_model.fit(
    adv_x,
    y,
    epochs=TRAINING_EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    verbose=1,
)

trained_regular = evaluate_accuracy(malware_model, x, y)
print("Trained model, regular data:", format_accuracy(trained_regular))

test_epsilon = train_epsilon
trained_adv_x = make_adversarial_examples(malware_model, x, test_epsilon)
trained_adversarial = evaluate_accuracy(malware_model, trained_adv_x, y)
print("Trained model, adversarial data:", format_accuracy(trained_adversarial))

results = {
    "untrained_regular": untrained_regular,
    "untrained_adversarial": untrained_adversarial,
    "trained_regular": trained_regular,
    "trained_adversarial": trained_adversarial,
}
print("\nReport:")
print_report(results)


**Student response:**

- Accuracy on adversarial samples after training:
- Accuracy on regular samples after training:
- Screenshot of accuracies:

## Steps 5-7: Compare Epsilon Values

Use the helper cells below instead of editing line numbers in `train.py`. Each experiment reloads a fresh copy of `MalwareClassifier.h5`, trains on adversarial examples generated with `train_epsilon`, and evaluates attacks generated with `test_epsilon`.

In [ ]:
# Reusable experiment function for different training and attack epsilons.
def run_adversarial_training_experiment(train_epsilon, test_epsilon, epochs=TRAINING_EPOCHS):
    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    model = load_compiled_model()
    regular_before = evaluate_accuracy(model, x, y)

    train_adv = make_adversarial_examples(model, x, train_epsilon)
    adversarial_before = evaluate_accuracy(model, train_adv, y)

    model.fit(
        train_adv,
        y,
        epochs=epochs,
        batch_size=BATCH_SIZE,
        shuffle=False,
        verbose=0,
    )

    regular_after = evaluate_accuracy(model, x, y)
    test_adv = make_adversarial_examples(model, x, test_epsilon)
    adversarial_after = evaluate_accuracy(model, test_adv, y)

    results = {
        "train_epsilon": train_epsilon,
        "test_epsilon": test_epsilon,
        "untrained_regular": regular_before,
        "untrained_adversarial": adversarial_before,
        "trained_regular": regular_after,
        "trained_adversarial": adversarial_after,
    }
    print(f"Training epsilon: {train_epsilon}")
    print(f"Testing epsilon:  {test_epsilon}")
    print_report(results)
    return results


In [ ]:
# Lab scenario helpers.
# Run these one at a time as you answer the lab questions.

# Step 5: train with epsilon 0.022, then test with smaller epsilon 0.005.
# smaller_attack_results = run_adversarial_training_experiment(train_epsilon=0.022, test_epsilon=0.005)

# Step 6: train with epsilon 0.022, then test with larger epsilon 0.1.
# larger_attack_results = run_adversarial_training_experiment(train_epsilon=0.022, test_epsilon=0.1)

# Step 7: train with larger epsilon 0.1, then test with smaller epsilon 0.005.
# large_training_small_attack_results = run_adversarial_training_experiment(train_epsilon=0.1, test_epsilon=0.005)


**Student response:**

- Step 5, train `0.022`, test `0.005`: accuracy and whether training was sufficient:
- Step 6, train `0.022`, test `0.1`: accuracy and whether training was sufficient:
- Step 7, train `0.1`, test `0.005`: adversarial accuracy, regular accuracy, tradeoff, and whether adversarial training is a good protection here: